In [6]:
import sys
import json
import re
import pandas as pd

def parse_line(line):
    # Regex to extract UNIX time (e.g., 1764000892.487560)
    unix_time_match = re.search(r'(\d+\.\d+)', line)
    if not unix_time_match:
        return None
    unix_time = float(unix_time_match.group(1))

    # Regex to extract the entire JSON object starting with "class":"TPV"
    json_match = re.search(r'\{.*?"class":"TPV".*?\}', line, re.DOTALL)
    if not json_match:
        return None
    json_str = json_match.group(0)

    try:
        data = json.loads(json_str)
    except json.JSONDecodeError:
        return None

    # Extract the required fields
    lat = data.get('lat')
    lon = data.get('lon')
    alt = data.get('alt')
    altMSL = data.get('altMSL')

    return {
        'unix_time': unix_time,
        'lat': lat,
        'lon': lon,
        'alt': alt,
        'altMSL': altMSL
    }

def main():
    filename = '/Users/thatch/Downloads/20251124_161604_gpspipe_stdout.log'  # Replace with your file path if needed

    results = []

    try:
        with open(filename, 'r') as f:
            for line in f:
                if 'TPV' in line:
                    result = parse_line(line)
                    if result:
                        results.append(result)
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return
    
    # Convert list of dicts to DataFrame
    df = pd.DataFrame(results)

    # Optional: Print the DataFrame or save it as CSV
    print(df)
    # Uncomment below to save as CSV
    # df.to_csv('output.csv', index=False)

if __name__ == '__main__':
    main()


       unix_time        lat        lon      alt   altMSL
0   1.764001e+09 -33.428637 -70.782027  493.507  493.507
1   1.764001e+09 -33.428638 -70.782027  493.017  493.017
2   1.764001e+09 -33.428639 -70.782026  492.717  492.717
3   1.764001e+09 -33.428641 -70.782026  492.081  492.081
4   1.764001e+09 -33.428643 -70.782025  491.809  491.809
..           ...        ...        ...      ...      ...
68  1.764001e+09 -33.428654 -70.781985  507.166  507.166
69  1.764001e+09 -33.428645 -70.781990  507.135  507.135
70  1.764001e+09 -33.428645 -70.781991  507.179  507.179
71  1.764001e+09 -33.428638 -70.781997  505.334  505.334
72  1.764001e+09 -33.428634 -70.781999  505.172  505.172

[73 rows x 5 columns]
